# S0 · Plantilla del cuaderno de la sesión
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/plantillas/S0_AC_CLASE_Colab_PLANTILLA.ipynb)

Este cuaderno muestra cómo están armados los cuadernos del curso. Antes de empezar:

1. **Archivo → Guardar una copia en Drive.** Trabaja siempre en tu copia.
2. Escribe tu nombre y tu código en la celda de identificación. **Tus datos dependen de tu código**: los de tus compañeros son distintos.
3. Completa cada hueco marcado con ✎. No cambies las demás líneas.
4. Al terminar: **Entorno de ejecución → Reiniciar y ejecutar todo**, revisa que no haya errores y entrega en Classroom.

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones 1 a 5: nombre, código, sesión, huella y fecha. Después, un renglón por resultado
    («resultado:<nombre>», valor). Luego un renglón vacío y la tabla: encabezados y datos.
    `columnas` es un diccionario {encabezado: lista de valores}, todas de la misma longitud.
    """
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["fecha", ahora()])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


import matplotlib.pyplot as plt

def completa(valor, ejercicio):
    if valor is None:
        raise ValueError(f'Completa el hueco ✎ del ejercicio {ejercicio} y vuelve a ejecutar.')
    return valor

print('Herramientas del curso listas.')

## Identificación

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S0"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Un registrador de temperatura convierte el intervalo de −20 a 50 °C con un convertidor de N bits, guarda hasta
32 000 lecturas y se programa con un intervalo de registro. **Tus valores son propios**: ejecuta la celda siguiente.

In [ ]:
RANGO_C = 70.0                                   # de −20 a 50 °C
N_BITS = asignar(alumno, [8, 10, 12], salto=0)
MEMORIA = 32000                                  # lecturas
INTERVALO_S = asignar(alumno, [10, 30, 60, 120, 300], salto=1)
PERIODO_MIN = parametro(alumno, "periodo", 8, 30, 1)   # periodo del ciclo del horno, en minutos
print(f"N = {N_BITS} bits · intervalo de registro = {INTERVALO_S} s · periodo del proceso = {PERIODO_MIN} min")

### ★ Ejercicio 1 · Resolución del registrador (3 pts)
¿Cuántos grados vale un escalón del convertidor (LSB)?  LSB = rango / 2ᴺ.

*Lo que predice en la práctica:* las lecturas sólo pueden cambiar en saltos de un LSB.

In [ ]:
lsb_C = None   # ✎ escribe aquí la expresión

print(f'LSB = {completa(lsb_C, 1):.4f} °C')

### Ejercicio 2 · Error máximo de cuantización (2 pts)
El error de redondear al escalón más cercano nunca pasa de medio LSB.

*Lo que predice en la práctica:* en la parte B, el error máximo observado no debe pasar de este valor.

In [ ]:
error_max_C = None   # ✎ escribe aquí la expresión

print(f'Error máximo = {completa(error_max_C, 2):.4f} °C')

### Ejercicio 3 · ¿Cuánto dura la memoria? (2 pts)
Duración = número de lecturas × intervalo. Exprésala en días (1 día = 86 400 s).

In [ ]:
duracion_dias = None   # ✎ escribe aquí la expresión

print(f'La memoria se llena en {completa(duracion_dias, 3):.2f} días')

### Ejercicio 4 · Intervalo máximo de registro (2 pts)
Para no perder el ciclo del proceso hay que registrar al menos dos veces por periodo (criterio de Nyquist).
Calcula el intervalo máximo en minutos y revisa la gráfica: los puntos naranjas son lo que guardaría el registrador.

In [ ]:
intervalo_max_min = None   # ✎ escribe aquí la expresión

completa(intervalo_max_min, 4)
t = np.linspace(0, 3 * PERIODO_MIN, 600)
T = 180 + 40 * np.sin(2 * np.pi * t / PERIODO_MIN)
tm = np.arange(0, 3 * PERIODO_MIN, intervalo_max_min)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t, T, label="proceso")
ax.plot(tm, 180 + 40 * np.sin(2 * np.pi * tm / PERIODO_MIN), "o", label=f"registro cada {intervalo_max_min:.1f} min")
ax.set_xlabel("tiempo (min)"); ax.set_ylabel("temperatura (°C)"); ax.legend()
titulo(ax, "Registro al límite de Nyquist", alumno)
firma(fig, alumno); plt.show()

### Cierre (1 pt)
Escribe en esta celda, con tus números: si tu registrador guardara una lectura cada intervalo máximo del ejercicio 4,
¿cuántos días duraría la memoria? ¿Te conviene ese intervalo? ✎

## Parte B · Práctica (80 min)
Temperatura real de Guadalajara, hora por hora, en una semana que depende de tu código
([Open-Meteo](https://open-meteo.com/), licencia CC BY 4.0). Si no hay internet, el cuaderno usa datos sintéticos y lo dice.
Vas a cuantizarla con **tu** convertidor y comparar el error obtenido con el que predijiste en el ejercicio 2.

In [ ]:
import json, urllib.request, datetime as dt
lunes = dt.date(2025, 1, 6) + dt.timedelta(weeks=int(alumno.codigo) % 50)
domingo = lunes + dt.timedelta(days=6)
url = ("https://archive-api.open-meteo.com/v1/archive?latitude=20.67&longitude=-103.35"
       f"&start_date={lunes}&end_date={domingo}&hourly=temperature_2m&timezone=America%2FMexico_City")
try:
    with urllib.request.urlopen(url, timeout=20) as r:
        d = json.load(r)["hourly"]
    fechas, T_real = d["time"], np.array(d["temperature_2m"], dtype=float)
    FUENTE = "Open-Meteo (ERA5), Guadalajara"
except Exception as e:
    h = np.arange(7 * 24)
    fechas = [f"hora {i}" for i in h]
    T_real = 20 + 7 * np.sin(2 * np.pi * (h - 9) / 24) + alumno.rng("ruido").normal(0, 0.6, h.size)
    FUENTE = "datos sintéticos (sin conexión)"
print(f"Semana del {lunes} al {domingo} · {len(T_real)} lecturas · fuente: {FUENTE}")

In [ ]:
# Cuantización con tu convertidor: −20 °C es la cuenta 0
cuentas = np.round((T_real + 20) / lsb_C)
T_cuant = cuentas * lsb_C - 20
error = T_cuant - T_real
error_obtenido_C = float(np.max(np.abs(error)))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
a1.plot(T_real, label="real"); a1.step(range(len(T_cuant)), T_cuant, where="mid", label=f"{N_BITS} bits")
a1.set_ylabel("T (°C)"); a1.legend(); titulo(a1, "Temperatura real y cuantizada", alumno)
a2.plot(error); a2.axhline(error_max_C, ls="--", c="r"); a2.axhline(-error_max_C, ls="--", c="r")
a2.set_ylabel("error (°C)"); a2.set_xlabel("lectura (h)")
RESULTADOS = {"lsb_C": lsb_C, "error_predicho_C": error_max_C, "error_obtenido_C": error_obtenido_C}
firma(fig, alumno, RESULTADOS); plt.show()
print(f"Error predicho {error_max_C:.4f} °C · error obtenido {error_obtenido_C:.4f} °C")

### CSV para tu hoja de resultados
La celda siguiente descarga `S0_<código>.csv`. En tu hoja de resultados: pestaña **CSV → Archivo → Importar →
Subir → Reemplazar la hoja actual**. La pestaña Resultados arma la tabla y la gráfica sola.

In [ ]:
ruta = f"S0_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"fecha": fechas, "T_real_C": T_real, "T_cuantizada_C": T_cuant, "error_C": error})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Cierre del cuaderno

In [ ]:
print(f"{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · huella {huella(alumno, RESULTADOS)}")